# Prompt Evaluations

In [14]:
# Importing Necessary Modules
import boto3
import json

In [15]:
# Creating the bedrock client
anthropic_bedrock = boto3.client('bedrock-runtime', region_name='us-west-2')

# Model ID
model_id = 'global.anthropic.claude-sonnet-4-6'

# System Prompt
system_prompt = ''

# Temperature 
temperature = 1.0

In [24]:
# Defining the function for user message
messages = []

def add_user_message(messages, text):
    user_message = {
        'role' : 'user',
        'content' : [
            { 'text' : text }
        ]
    }

    messages.append(user_message)
    return user_message

# Defining the function for assistance message
def add_assistant_message(messages, text):
    assistant_message = {
        'role' : 'assistant',
        'content' : [
            { 'text' : text }
        ]
    }

    messages.append(assistant_message)

# Function to invoke the model and return the assistant response
def chat(messages, system=None, temperature = 1.0, stop_sequences=[]):

    params = {
        "modelId" : model_id,
        "messages" : messages,
        "inferenceConfig" : {
            'maxTokens' : 512,
            'temperature' : temperature,
            'stopSequences' : stop_sequences
        }
    }

    if system:
        params["system"] = [{'text' : system}]

    response = anthropic_bedrock.converse(**params)

    return response['output']['message']['content'][0]['text']

## Creating dataset for prompt evaluation

In [43]:
# Function to generate the dataset
def generate_dataset():
    
    prompt = """
    Generate 3 AWS-related tasks that require Python, JSON, or Regex Solutions
    Focus on tasks that can be solved by writing a single Python function, a single JSON object, or tasks that do not require writing much code.
    
    Example output:
    [
        {
            "task": "Description of task"
        }
    ]

    Please generate 3 Objects.
    """

    prompt_messages = []
    add_user_message(prompt_messages, prompt)
    text = chat(prompt_messages)
    
    print(f"Raw response: {repr(text)}")
    print(f"Response length: {len(text)}")
    
    return json.loads(text)

dataset = generate_dataset()

Raw response: '```json\n[\n    {\n        "task": "Write a Python function that takes an AWS S3 bucket name and a file prefix as input, connects to S3 using boto3, and returns a list of all object keys that match the given prefix, along with their sizes in a dictionary format like {\'key\': \'example/file.txt\', \'size_kb\': 12.4}."\n    },\n    {\n        "task": "Write a JSON object representing an AWS IAM policy that allows a Lambda function to read objects from a specific S3 bucket, write logs to CloudWatch, and publish messages to an SNS topic, following the least privilege principle with proper Resource ARN restrictions."\n    },\n    {\n        "task": "Write a Python function using the regex module that parses an AWS CloudWatch log line and extracts the timestamp, log level, request ID, and message into a structured dictionary. The log format looks like: \'2024-01-15T10:23:45.123Z ERROR [RequestId: abc-123-def] Something went wrong\'."\n    }\n]\n```'
Response length: 937


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [39]:
# Function to generate the dataset
def generate_dataset():
    
    prompt = """
    Generate 3 AWS-related tasks that require Python, JSON, or Regex Solutions
    Focus on tasks that can be solved by writing a single Python function, a single JSON object, or tasks that do not require writing much code.
    
    Example output:
    [
        {
            "task": "Description of task"
        }
    ]

    Please generate 3 Objects.
    """

    prompt_messages = []
    add_user_message(prompt_messages, prompt)
    text = chat(prompt_messages)
    if text.startswith(''):
        text = text.split('\n', 1)[1]  # Remove first line (json)
        text = text.rsplit('\n', 1)[0]  # Remove last line ()
        
    return json.loads(text)

In [40]:
dataset = generate_dataset()

In [41]:
dataset

[{'task': "Write a Python function that takes an AWS S3 bucket name and a file prefix as input, connects to S3 using boto3, and returns a list of all object keys that match the given prefix, along with their file sizes in a dictionary format like {'key': 'size'}."},
 {'task': "Create a JSON object that represents a valid AWS IAM policy document which grants read-only access (GetObject, ListBucket) to a specific S3 bucket named 'my-company-data', while explicitly denying the ability to delete any objects from that same bucket."},
 {'task': 'Write a Python function using the `re` module that validates and parses an AWS ARN (Amazon Resource Name) string, extracting its components (partition, service, region, account-id, and resource) into a dictionary, and returns None if the ARN format is invalid.'}]

In [42]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

## Running the Eval

In [44]:
# This function takes a test case and merges it with our prompt template:
def run_prompt(test_case):
    "Merges the prompt and test case input, then return the result"""

    prompt = f"""
    Please solve the following task:
    {test_case['task']}
    """

    eval_messages = []
    add_user_message(eval_messages, prompt)
    output = chat(eval_messages)

    return output

In [45]:
# This function orchestrates running a single test case and grading the result:
def run_test_case(test_case):

    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [51]:
# This is the main orchestrator that processes the entire dataset:
def run_eval(dataset):

    """Load the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [52]:
# To execute our evaluation pipeline, we load the dataset and call our main function:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [53]:
# Once the evaluation completes, you can inspect the results with formatted JSON output:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Object Listing Function\n\n## Solution\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError, NoCredentialsError\nfrom typing import Optional\n\n\ndef list_s3_objects_with_sizes(\n    bucket_name: str,\n    prefix: str = \"\",\n    aws_access_key_id: Optional[str] = None,\n    aws_secret_access_key: Optional[str] = None,\n    region_name: Optional[str] = None,\n) -> dict[str, int]:\n    \"\"\"\n    Lists all S3 objects matching a given prefix and returns their keys with sizes.\n\n    Args:\n        bucket_name (str): The name of the S3 bucket.\n        prefix (str): The file prefix to filter objects. Defaults to \"\" (all objects).\n        aws_access_key_id (str, optional): AWS access key ID. Uses default credentials if None.\n        aws_secret_access_key (str, optional): AWS secret access key. Uses default credentials if None.\n        region_name (str, optional): AWS region name. Uses default region if None.\n\n    Returns:\n        d